# Split Datasets

Split dataset to Train/Validation/Test sets

## A. Overview

Random split to Train, Validation and Test sets

In [1]:
%pip install datasketch

Note: you may need to restart the kernel to use updated packages.


## B. Combine Datasets

In [ ]:
from pathlib import Path
import csv
import os
import random
import json
import re
from collections import defaultdict

from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from datasketch import MinHash, MinHashLSH

import configuration
from src import data_utils, setup
from src.normalizer import similarity

RANDS_STATE = 42
random.seed(RANDS_STATE)

dataset_path = Path('..') / 'data'

### B.1. Load Datasets

In [3]:
# disaster_frac = data_utils.get_data_disaster_fraction()
# disaster_filepath = dataset_path / 'disaster' / str(disaster_frac)

# df_disaster_informative = pd.read_csv(disaster_filepath / 'informative.csv')
# df_disaster_informative['subset'] = 'disaster'
# # df_disaster_humanitarian = pd.read_csv(disaster_filepath / 'humanitarian.csv')
# # df_disaster_humanitarian['subset'] = 'disaster'

In [ ]:
extended_filepath = dataset_path / "extended"

df_weather = pd.read_csv(
    extended_filepath / "weather.csv"
)["tweet_text"].to_frame()
df_weather["informative"] = False
df_weather["subset"] = "weather"

df_out_topic = pd.read_csv(
    extended_filepath / "out_topic.csv"
)["tweet_text"].to_frame()
df_out_topic["informative"] = False
df_out_topic["subset"] = "out_topic"

### B.2. Combine Datasets

In [5]:
# df_informative = (
#     pd.concat([df_disaster_informative, df_weather, df_out_topic], ignore_index=True)
#     .sample(frac=1, random_state=setup.RANDOM_SEED)
#     .reset_index(drop=True)
# )
# df_informative.head()

## C. Filtering

In [ ]:
df_weather_processed = similarity.filter_duplicates_with_resume(df_weather, similarity_threshold=0.75) # 41414

Original dataset size: 47518
Running exact duplicate pre-pass...
Removed 18 exact duplicates before approximate matching.
Building MinHash LSH index and scanning for near-duplicates...
Final dataset size after near-duplicate removal: 45733


In [ ]:
# The df_out_topic is too large
# split to multiple partitions
# accept similar records accross them
partition_records = 1_000_000

df_out_topic_processed = None

for i in range(0, len(df_out_topic) // partition_records):
    start_idx = i * partition_records
    end_idx = min((i + 1) * partition_records, len(df_out_topic))
    partition = df_out_topic.iloc[start_idx:end_idx]
    partition_processed = similarity.filter_duplicates_with_resume(partition)
    # if i == 0:
    #     df_out_topic_processed = partition_processed
    # else:
    df_out_topic_processed = pd.concat([df_out_topic_processed, partition_processed], ignore_index=True)

In [ ]:
# df_out_topic_processed = similarity.filter_duplicates_with_resume(df_out_topic) # 41414

Original dataset size: 10403525
Running exact duplicate pre-pass...
Removed 21260 exact duplicates before approximate matching.
Building MinHash LSH index and scanning for near-duplicates...
Final dataset size after near-duplicate removal: 10233915


In [ ]:
# df_out_topic_processed.to_csv(extended_filepath / "out_topic_deduplicated.csv", index=False, quoting=csv.QUOTE_ALL)